### Libraries

In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


# Speech and Topic

We have a topic assigned to each paragraph from the clustering step.

Given that each paragraph can have multiple sentences, we can split the paragraph into sentences and perform NLI on each sentence against the topic. This way, we can capture more fine-grained stances within a paragraph.

Why not performing clustering on sentences instead of paragraphs? 
- Because we want to maintain the context of the paragraph, which can be important for understanding the stance. Clustering on sentences might lead to losing this context and could result in less coherent topics.

In [3]:
df = pd.read_json("speeches_with_topics.jsonl", lines=True)
df

,Speech_ID,Paragraph_ID,TEXT_SPAN,role,language_of_text,name,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document
0,81895,81895_p1,Aber es geht hier noch um andere Fragen. Es ge...,Member of the federal council,DE,Schmid Samuel,108,"108_National Security, Armed Forces, Defense","[National Security, Armed Forces, Defense]",[Der Bundesrat hat zudem am 21. September 2012...,"National Security, Armed Forces, Defense",False
1,81895,81895_p2,"Verschiedene andere Fragen, beispielsweise jen...",Member of the federal council,DE,Schmid Samuel,108,"108_National Security, Armed Forces, Defense","[National Security, Armed Forces, Defense]",[Der Bundesrat hat zudem am 21. September 2012...,"National Security, Armed Forces, Defense",False
2,81895,81895_p3,"Also sind wir bereit, die Analysen zu machen, ...",Member of the federal council,DE,Schmid Samuel,108,"108_National Security, Armed Forces, Defense","[National Security, Armed Forces, Defense]",[Der Bundesrat hat zudem am 21. September 2012...,"National Security, Armed Forces, Defense",False
3,83146,83146_p2,Zum Antrag der Minderheit Maury Pasquier: Dies...,For the committee,DE,Schwaller Urs,67,67_Health insurance costs,[Health insurance costs],[Zu den Anträgen im ersten Block: Bei Artikel ...,Health insurance costs,False
4,83146,83146_p3,Noch eine abschliessende Bemerkung: Wir streic...,For the committee,DE,Schwaller Urs,67,67_Health insurance costs,[Health insurance costs],[Zu den Anträgen im ersten Block: Bei Artikel ...,Health insurance costs,False
...,...,...,...,...,...,...,...,...,...,...,...,...
210223,288681,288681_p8,"En sus, la commission a adopté, par 13 voix co...",For the committee,FR,Python Valentine,149,149_Agriculture and Pesticides,[Agriculture and Pesticides],[Vous avez vu que le Conseil fédéral propose d...,Agriculture and Pesticides,False
210224,288681,288681_p9,Considérant le risque important de contaminati...,For the committee,FR,Python Valentine,68,68_Tobacco and alcohol regulation,[Tobacco and alcohol regulation],[La présente motion du Conseil des Etats conce...,Tobacco and alcohol regulation,False
210225,288681,288681_p11,"En commission, une minorité avait, quant à ell...",For the committee,FR,Python Valentine,68,68_Tobacco and alcohol regulation,[Tobacco and alcohol regulation],[La présente motion du Conseil des Etats conce...,Tobacco and alcohol regulation,False
210226,288681,288681_p12,La proposition individuelle Bäumle déposée hie...,For the committee,FR,Python Valentine,149,149_Agriculture and Pesticides,[Agriculture and Pesticides],[Vous avez vu que le Conseil fédéral propose d...,Agriculture and Pesticides,False


### Split into sentences

In [4]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /home/matteo/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
punkt_mapping = {
    "FR": "french",
    "DE": "german",
    "IT": "italian",
}

In [6]:
df["LANG"] = df["language_of_text"].map(punkt_mapping)

In [7]:
rows_list = []
for i, row in tqdm(df.iterrows(), total=len(df), desc="Splitting into sentences"):
    sentences = nltk.sent_tokenize(row["TEXT_SPAN"], language=row["LANG"])
    for sent_id, sentence in enumerate(sentences):
        rows_list.append({
            "Speech_ID": row["Speech_ID"],
            "Paragraph_ID": row["Paragraph_ID"],
            "Sentence_ID": sent_id,
            "Text": sentence,
            "role": row["role"],
            "language_of_text": row["language_of_text"],
            "name": row["name"],
            "Representation": row["Representation"],
        })

sentences_df = pd.DataFrame(rows_list)
sentences_df

Splitting into sentences:   0%|          | 0/210228 [00:00<?, ?it/s]

,Speech_ID,Paragraph_ID,Sentence_ID,Text,role,language_of_text,name,Representation
0,81895,81895_p1,0,Aber es geht hier noch um andere Fragen.,Member of the federal council,DE,Schmid Samuel,"[National Security, Armed Forces, Defense]"
1,81895,81895_p1,1,"Es geht insbesondere auch darum, anhand der De...",Member of the federal council,DE,Schmid Samuel,"[National Security, Armed Forces, Defense]"
2,81895,81895_p1,2,In den Schulen sind jetzt die geburtenstarken ...,Member of the federal council,DE,Schmid Samuel,"[National Security, Armed Forces, Defense]"
3,81895,81895_p1,3,"Das heisst, dass sich die geburtenschwachen Ja...",Member of the federal council,DE,Schmid Samuel,"[National Security, Armed Forces, Defense]"
4,81895,81895_p1,4,In ein paar Jahren sind sie bei mir.,Member of the federal council,DE,Schmid Samuel,"[National Security, Armed Forces, Defense]"
...,...,...,...,...,...,...,...,...
866544,288681,288681_p8,0,"En sus, la commission a adopté, par 13 voix co...",For the committee,FR,Python Valentine,[Agriculture and Pesticides]
866545,288681,288681_p9,0,Considérant le risque important de contaminati...,For the committee,FR,Python Valentine,[Tobacco and alcohol regulation]
866546,288681,288681_p11,0,"En commission, une minorité avait, quant à ell...",For the committee,FR,Python Valentine,[Tobacco and alcohol regulation]
866547,288681,288681_p12,0,La proposition individuelle Bäumle déposée hie...,For the committee,FR,Python Valentine,[Agriculture and Pesticides]


# Hypothesis Template

In [8]:
hp_df = sentences_df.copy()

In [9]:
TEMPLATE = "The speaker advocates in favor of {}."

In [10]:
hp_df["hypothesis"] = hp_df["Representation"].apply(lambda x: TEMPLATE.format(x))
hp_df["hypothesis"].values

<ArrowStringArray>
['The speaker advocates in favor of ['National Security, Armed Forces, Defense'].',
 'The speaker advocates in favor of ['National Security, Armed Forces, Defense'].',
 'The speaker advocates in favor of ['National Security, Armed Forces, Defense'].',
 'The speaker advocates in favor of ['National Security, Armed Forces, Defense'].',
 'The speaker advocates in favor of ['National Security, Armed Forces, Defense'].',
 'The speaker advocates in favor of ['National Security, Armed Forces, Defense'].',
 'The speaker advocates in favor of ['National Security, Armed Forces, Defense'].',
 'The speaker advocates in favor of ['National Security, Armed Forces, Defense'].',
 'The speaker advocates in favor of ['National Security, Armed Forces, Defense'].',
 'The speaker advocates in favor of ['National Security, Armed Forces, Defense'].',
 ...
           'The speaker advocates in favor of ['Tobacco and alcohol regulation'].',
           'The speaker advocates in favor of ['Toba

# NLI Model

In [15]:
model_name = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(251000, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [12]:
import numpy as np
batch_size = 1
n_rows = hp_df.shape[0]
entailment_scores = np.zeros(n_rows)
neutral_scores = np.zeros(n_rows)
contradiction_scores = np.zeros(n_rows)

In [ ]:
model.to(device)
model.eval()
for i in tqdm(range(0, len(hp_df), batch_size), desc="Processing batches"):
    batch_premises = hp_df["Text"].iloc[i : i + batch_size].tolist()
    batch_hypotheses = hp_df["hypothesis"].iloc[i : i + batch_size].tolist()

    inputs = tokenizer(
        batch_premises,
        batch_hypotheses,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)

    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

    probs = probs.cpu().tolist()

    # Model mapping: 0: entailment, 1: neutral, 2: contradiction
    for j, prob in enumerate(probs):
        entailment_scores[i + j] = prob[0]
        neutral_scores[i + j] = prob[1]
        contradiction_scores[i + j] = prob[2]

Processing batches:   0%|          | 0/866549 [00:00<?, ?it/s]

[[0.45654296875, 0.54345703125]]


IndexError: list index out of range

In [56]:
for scores in zip(entailment_scores, neutral_scores, contradiction_scores):
    print(f"Entailment: {scores[0]:.4f}, Neutral: {scores[1]:.4f}, Contradiction: {scores[2]:.4f}")
    break

Entailment: 0.0203, Neutral: 0.9722, Contradiction: 0.0077


In [58]:
# Assign scores back to DataFrame
hp_df["Score_Support"] = entailment_scores
hp_df["Score_Neutral"] = neutral_scores
hp_df["Score_Oppose"] = contradiction_scores
hp_df.head()

,Speech_ID,Paragraph_ID,Sentence_ID,Text,role,language_of_text,name,Representation,hypothesis,Score_Support,Score_Neutral,Score_Oppose,Final_Stance
0,81895,81895_p1,0,Aber es geht hier noch um andere Fragen.,Member of the federal council,DE,Schmid Samuel,"National Security, Armed Forces, Defense",The speaker advocates in favor of National Sec...,0.020325,0.972168,0.007687,Neutral
1,81895,81895_p1,1,"Es geht insbesondere auch darum, anhand der De...",Member of the federal council,DE,Schmid Samuel,"National Security, Armed Forces, Defense",The speaker advocates in favor of National Sec...,0.136597,0.861328,0.002182,Neutral
2,81895,81895_p1,2,In den Schulen sind jetzt die geburtenstarken ...,Member of the federal council,DE,Schmid Samuel,"National Security, Armed Forces, Defense",The speaker advocates in favor of National Sec...,0.666016,0.332764,0.001073,Neutral
3,81895,81895_p1,3,"Das heisst, dass sich die geburtenschwachen Ja...",Member of the federal council,DE,Schmid Samuel,"National Security, Armed Forces, Defense",The speaker advocates in favor of National Sec...,0.256592,0.734375,0.009102,Neutral
4,81895,81895_p1,4,In ein paar Jahren sind sie bei mir.,Member of the federal council,DE,Schmid Samuel,"National Security, Armed Forces, Defense",The speaker advocates in favor of National Sec...,0.344482,0.646973,0.008522,Neutral


In [60]:
def determine_stance(row):
    if row["Score_Support"] > 0.75:
        return "Support"
    elif row["Score_Oppose"] > 0.75:
        return "Oppose"
    else:
        return "Neutral"

In [61]:
hp_df["Final_Stance"] = hp_df.apply(determine_stance, axis=1)
hp_df

,Speech_ID,Paragraph_ID,Sentence_ID,Text,role,language_of_text,name,Representation,hypothesis,Score_Support,Score_Neutral,Score_Oppose,Final_Stance
0,81895,81895_p1,0,Aber es geht hier noch um andere Fragen.,Member of the federal council,DE,Schmid Samuel,"National Security, Armed Forces, Defense",The speaker advocates in favor of National Sec...,0.020325,0.972168,0.007687,Neutral
1,81895,81895_p1,1,"Es geht insbesondere auch darum, anhand der De...",Member of the federal council,DE,Schmid Samuel,"National Security, Armed Forces, Defense",The speaker advocates in favor of National Sec...,0.136597,0.861328,0.002182,Neutral
2,81895,81895_p1,2,In den Schulen sind jetzt die geburtenstarken ...,Member of the federal council,DE,Schmid Samuel,"National Security, Armed Forces, Defense",The speaker advocates in favor of National Sec...,0.666016,0.332764,0.001073,Neutral
3,81895,81895_p1,3,"Das heisst, dass sich die geburtenschwachen Ja...",Member of the federal council,DE,Schmid Samuel,"National Security, Armed Forces, Defense",The speaker advocates in favor of National Sec...,0.256592,0.734375,0.009102,Neutral
4,81895,81895_p1,4,In ein paar Jahren sind sie bei mir.,Member of the federal council,DE,Schmid Samuel,"National Security, Armed Forces, Defense",The speaker advocates in favor of National Sec...,0.344482,0.646973,0.008522,Neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...
866544,288681,288681_p8,0,"En sus, la commission a adopté, par 13 voix co...",For the committee,FR,Python Valentine,Agriculture and Pesticides,The speaker advocates in favor of Agriculture ...,0.000000,0.000000,0.000000,Neutral
866545,288681,288681_p9,0,Considérant le risque important de contaminati...,For the committee,FR,Python Valentine,Tobacco and alcohol regulation,The speaker advocates in favor of Tobacco and ...,0.000000,0.000000,0.000000,Neutral
866546,288681,288681_p11,0,"En commission, une minorité avait, quant à ell...",For the committee,FR,Python Valentine,Tobacco and alcohol regulation,The speaker advocates in favor of Tobacco and ...,0.000000,0.000000,0.000000,Neutral
866547,288681,288681_p12,0,La proposition individuelle Bäumle déposée hie...,For the committee,FR,Python Valentine,Agriculture and Pesticides,The speaker advocates in favor of Agriculture ...,0.000000,0.000000,0.000000,Neutral


In [64]:
hp_df[hp_df["Final_Stance"] != "Neutral"][["Text", "hypothesis", "Score_Support", "Score_Neutral", "Score_Oppose", "Final_Stance"]].head(10)

,Text,hypothesis,Score_Support,Score_Neutral,Score_Oppose,Final_Stance
21,Weiter mischen sich die Kantone nun auch in de...,The speaker advocates in favor of Federalism a...,0.933594,0.062408,0.003883,Support
31,"On est en pleine étude sur le sujet, et le sys...",The speaker advocates in favor of Health insur...,0.852051,0.145630,0.002512,Support
53,"Monsieur le président de la Confédération, il ...",The speaker advocates in favor of Federal-Cant...,0.826660,0.171387,0.001816,Support
71,Des accords semblables ont déjà été signés ave...,The speaker advocates in favor of Internationa...,0.963379,0.035217,0.001218,Support
72,"Cet accord a une certaine importance, compte d...",The speaker advocates in favor of Internationa...,0.820312,0.174561,0.005325,Support
73,Une manifestation de ce genre attire l'attenti...,The speaker advocates in favor of Internationa...,0.889160,0.095581,0.015228,Support
79,"Nach Frankreich, Italien und Deutschland wurde...",The speaker advocates in favor of Migration an...,0.981934,0.013016,0.005222,Support
80,"Die Schweiz ist bestrebt, mit den umliegenden ...",The speaker advocates in favor of Migration an...,0.798340,0.139771,0.061981,Support
99,"Pour toutes ces raisons, nous vous invitons à ...",The speaker advocates in favor of Rejection of...,0.991211,0.004642,0.004341,Support
108,"D'autre part, les contribuables qui auraient u...",The speaker advocates in favor of Pension refo...,0.884766,0.114380,0.000818,Support


In [66]:
idx = 53
hp_df.iloc[idx][["Text", "hypothesis"]].values

array(["Monsieur le président de la Confédération, il faut que je vous dise que, suite à la décision qui a été prise par le Conseil national, c'est la commission de notre conseil qui, par mon intermédiaire, s'est adressée à la Conférence suisse des directrices et directeurs cantonaux de la santé pour lui demander si elle avait vu que, d'après la solution choisie par le Conseil national, les cantons devraient assumer 55 pour cent du financement.",
       'The speaker advocates in favor of Federal-Cantonal Relations.'],
      dtype=object)

In [ ]:
# Save the rich dataset
hp_df.to_json("speeches_with_stances.jsonl", orient="records", lines=True)

print(hp_df["Final_Stance"].value_counts())